In [1]:
!pip install -q flask flask-cors pyngrok pymupdf transformers torch

In [2]:
import os
os.environ["NGROK_AUTHTOKEN"] = "YOUR_NGROKAUTHTOKEN"

In [4]:
"""
LoanGuard AI

"""

from __future__ import annotations

import json
import logging
import os
import re
import time
import traceback
from dataclasses import dataclass, asdict, field
from typing import Any, Dict, List, Optional, Tuple

import fitz  # PyMuPDF
import torch
from flask import Flask, jsonify, request
from flask_cors import CORS
from pyngrok import ngrok
from transformers import AutoModelForCausalLM, AutoTokenizer

# LOGGING
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("loanguard")

# CONFIG
class Config:
    MODEL_NAME: str = os.getenv("LG_MODEL", "Qwen/Qwen2.5-0.5B-Instruct")
    NGROK_TOKEN: str = os.getenv("NGROK_AUTHTOKEN", "")
    PORT: int = int(os.getenv("PORT", "5000"))
    MAX_LLM_CHARS: int = int(os.getenv("MAX_LLM_CHARS", "12000"))
    MAX_NEW_TOKENS: int = int(os.getenv("MAX_NEW_TOKENS", "512"))
    MAX_PDF_PAGES: int = int(os.getenv("MAX_PDF_PAGES", "60"))
    MAX_UPLOAD_MB: int = int(os.getenv("MAX_UPLOAD_MB", "32"))
    LATE_MISS_ESTIMATE: int = 3
    BOUNCE_MISS_ESTIMATE: int = 2
    PENAL_OVERDUE_MONTHS: int = 2
    INSURANCE_RATE: float = 0.015
    HIDDEN_CHARGES_ESTIMATE: int = 3000


cfg = Config()


# DATA MODELS
@dataclass
class ExtractedData:
    loan_amount: int = 0
    emi_amount: int = 0
    interest_rate_percent: float = 0.0
    tenure_months: int = 0

    processing_fee_percent: float = 0.0
    processing_fee_amount: int = 0
    prepayment_penalty_percent: float = 0.0
    foreclosure_charge_percent: float = 0.0
    late_payment_fee: int = 0
    bounce_charge: int = 0
    penal_interest_percent_monthly: float = 0.0
    penal_interest_percent_annual: float = 0.0

    has_prepayment_penalty: bool = False
    has_foreclosure_charges: bool = False
    has_hidden_charges: bool = False
    auto_debit_mandatory: bool = False
    auto_debit_optional: bool = False
    has_insurance_addon: bool = False

    has_rate_reset_clause: bool = False
    has_legal_cost_transfer: bool = False
    has_cross_default_clause: bool = False


@dataclass
class CostBreakdown:
    processing_fee_cost: int = 0
    prepayment_cost: int = 0
    foreclosure_cost: int = 0
    late_payment_cost_estimate: int = 0
    bounce_charge_cost_estimate: int = 0
    penal_interest_cost_estimate: int = 0
    insurance_addon_cost_estimate: int = 0
    hidden_charges_estimate: int = 0
    total_estimated_cost: int = 0


@dataclass
class Clause:
    title: str
    type: str
    severity: str
    plain_language_en: str
    financial_impact_rupees: int = 0
    recommendation: str = ""
    evidence_text: str = ""
    confidence: float = 0.95


@dataclass
class AnalysisResult:
    document_name: str
    engine_version: str = "loanguard-ui-v4.2"
    processing_time_sec: float = 0.0

    confidence_score: float = 0.0

    overall_score: int = 0
    overall_risk_level: str = "Unknown"
    overall_summary: str = ""
    total_hidden_cost_estimate_rupees: int = 0

    borrower_signal: str = ""

    risk_reasons: List[str] = field(default_factory=list)

    metrics: Dict[str, Any] = field(default_factory=dict)
    extracted_data: Dict[str, Any] = field(default_factory=dict)
    cost_breakdown: Dict[str, Any] = field(default_factory=dict)
    clauses: List[Dict[str, Any]] = field(default_factory=list)
    decision_ready_report: str = ""
    warnings: List[str] = field(default_factory=list)

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


# MODEL SINGLETON
class ModelSingleton:
    _tokenizer: Optional[AutoTokenizer] = None
    _model: Optional[AutoModelForCausalLM] = None
    _device: str = "cpu"

    @classmethod
    def load(cls) -> None:
        if cls._model is not None:
            return

        cls._device = "cuda" if torch.cuda.is_available() else "cpu"
        dtype = torch.float16 if cls._device == "cuda" else torch.float32

        log.info("Loading model %s on %s", cfg.MODEL_NAME, cls._device)

        cls._tokenizer = AutoTokenizer.from_pretrained(
            cfg.MODEL_NAME,
            trust_remote_code=True,
        )

        cls._model = AutoModelForCausalLM.from_pretrained(
            cfg.MODEL_NAME,
            dtype=dtype,
            device_map="auto" if cls._device == "cuda" else None,
            trust_remote_code=True,
        )

        if cls._device == "cpu":
            cls._model = cls._model.to(cls._device)

        log.info("✅ Model ready on %s", cls._device)

    @classmethod
    def generate(cls, messages: List[Dict[str, str]], max_new_tokens: int = 512) -> str:
        if cls._model is None or cls._tokenizer is None:
            raise RuntimeError("Model not loaded.")

        prompt = cls._tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        inputs = cls._tokenizer(prompt, return_tensors="pt").to(cls._model.device)

        with torch.no_grad():
            output = cls._model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.08,
                pad_token_id=cls._tokenizer.eos_token_id,
            )

        new_tokens = output[0][inputs["input_ids"].shape[1]:]
        return cls._tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# HELPERS
def normalize(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()


def clamp(val: float, lo: float, hi: float) -> float:
    return max(lo, min(val, hi))


def rupees(value: Any) -> int:
    try:
        return int(round(float(value))) if value else 0
    except (TypeError, ValueError):
        return 0


def parse_json_safe(text: str) -> Dict[str, Any]:
    text = re.sub(r"```(?:json)?", "", text).strip()
    match = re.search(r"\{[\s\S]*\}", text)
    if not match:
        raise ValueError("No JSON object found in model output")
    raw = match.group(0)
    raw = re.sub(r",\s*([}\]])", r"\1", raw)
    try:
       return json.loads(raw)
    except:
       return {}


def first_amount(text: str, patterns: List[str]) -> float:
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            raw = m.group(1).replace(",", "").strip()
            try:
                return float(raw)
            except ValueError:
                continue
    return 0.0


def contains_any(text: str, patterns: List[str]) -> bool:
    return any(re.search(p, text, re.IGNORECASE) for p in patterns)


def extract_snippet(text: str, patterns: List[str], window: int = 180) -> str:
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            start = max(0, m.start() - window)
            end = min(len(text), m.end() + window)
            return normalize(text[start:end])
    return ""


def severity_rank(sev: str) -> int:
    order = {"High": 0, "Medium": 1, "Low": 2}
    return order.get(sev, 9)


# PDF EXTRACTION
def extract_pdf_text(pdf_bytes: bytes, max_pages: int = cfg.MAX_PDF_PAGES) -> str:
    try:
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
    except Exception as exc:
        raise ValueError(f"Cannot open PDF: {exc}") from exc

    pages = []
    page_count = min(len(doc), max_pages)

    for i in range(page_count):
        page = doc[i]
        text = page.get_text("text", sort=True).strip()
        if not text:
            blocks = page.get_text("blocks", sort=True)
            text = "\n".join(b[4] for b in blocks if isinstance(b[4], str)).strip()
        pages.append(text)

    doc.close()

    full_text = "\n\n".join(pages).strip()
    if len(full_text) < 200:
        raise ValueError("PDF contains too little readable text.")
    return full_text


# AI EXTRACTION (PRIMARY)
class AIExtractor:
    SYSTEM = """
You are an expert financial contract analysis AI for Indian loan agreements.

Your tasks:
1. Read dense legal loan agreements carefully
2. Extract key financial and penalty clauses
3. Detect both presence and explicit absence of risky terms
4. Strictly do not hallucinate
5. If a term is not mentioned, default conservatively to false or 0
6. Do NOT mention EMI or monthly repayment unless explicitly provided.
Very important:
- "No prepayment penalty" means has_prepayment_penalty = false
- "No foreclosure charges" means has_foreclosure_charges = false
- "No hidden charges" means has_hidden_charges = false
- "Auto-debit optional" means auto_debit_mandatory = false and auto_debit_optional = true

Return ONLY strict JSON.
"""

    SCHEMA = """{
  "loan_amount": 0,
  "emi_amount": 0,
  "interest_rate_percent": 0,
  "tenure_months": 0,
  "processing_fee_percent": 0,
  "processing_fee_amount": 0,
  "prepayment_penalty_percent": 0,
  "foreclosure_charge_percent": 0,
  "late_payment_fee": 0,
  "bounce_charge": 0,
  "penal_interest_percent_monthly": 0,
  "penal_interest_percent_annual": 0,
  "has_prepayment_penalty": false,
  "has_foreclosure_charges": false,
  "has_hidden_charges": false,
  "auto_debit_mandatory": false,
  "auto_debit_optional": false,
  "has_insurance_addon": false,
  "has_rate_reset_clause": false,
  "has_legal_cost_transfer": false,
  "has_cross_default_clause": false
}"""

    def extract(self, text: str) -> Dict[str, Any]:
        snippet = text[:cfg.MAX_LLM_CHARS]
        messages = [
            {"role": "system", "content": self.SYSTEM},
            {
                "role": "user",
                "content": f"SCHEMA:\n{self.SCHEMA}\n\nTEXT:\n{snippet}",
            },
        ]
        raw = ModelSingleton.generate(messages, max_new_tokens=cfg.MAX_NEW_TOKENS)
        return parse_json_safe(raw)


# LIGHT VALIDATION LAYER
class ValidationEngine:
    _INR = r"(?:rs\.?|inr|₹)\s*([\d,]+(?:\.\d+)?)"

    @staticmethod
    def _inr_patterns(*prefixes: str) -> List[str]:
        return [rf"{p}[^₹\d]{{0,40}}{ValidationEngine._INR}" for p in prefixes]

    def extract_rule_hints(self, text: str) -> ExtractedData:
        t = text.lower()
        d = ExtractedData()

        d.loan_amount = rupees(first_amount(t, [
            *self._inr_patterns(
                "loan amount", "sanctioned amount", "principal amount",
                "lend a sum of", "sum of"
            ),
            r"(?:rs\.?|inr|₹)\s*([\d,]+(?:\.\d+)?)\s+(?:only\b|\/\-)",
        ]))

        d.emi_amount = rupees(first_amount(t, [
            *self._inr_patterns("emi amount", "monthly emi", "monthly instalment", "instalment of"),
            r"(?:rs\.?|inr|₹)\s*([\d,]+(?:\.\d+)?)\s+each",
        ]))

        m = re.search(r"interest rate[^%]{0,60}?([\d.]+)\s*%", t, re.IGNORECASE)
        if m:
            d.interest_rate_percent = float(m.group(1))

        m = re.search(r"tenure[^0-9]{0,20}(\d+)\s*(month|year)", t, re.IGNORECASE)
        if m:
            val = int(m.group(1))
            d.tenure_months = val if "month" in m.group(2) else val * 12

        d.processing_fee_percent = first_amount(t, [
            r"processing fee[^%]{0,60}?([\d.]+)\s*%",
            r"non[- ]refundable processing fee[^%]{0,60}?([\d.]+)\s*%",
        ])
        d.processing_fee_amount = rupees(first_amount(t, self._inr_patterns(
            "processing fee", "processing fees", "processing charge"
        )))

        d.prepayment_penalty_percent = first_amount(t, [
            r"prepayment (?:penalty|charge)[^%]{0,60}?([\d.]+)\s*%",
            r"pre-?payment (?:penalty|charge)[^%]{0,60}?([\d.]+)\s*%",
        ])

        d.foreclosure_charge_percent = first_amount(t, [
            r"foreclosure charges?[^%]{0,60}?([\d.]+)\s*%",
        ])

        d.late_payment_fee = rupees(first_amount(t, [
            *self._inr_patterns("late payment fee", "late payment charge", "overdue charge", "delay penalty"),
            r"(?:rs\.?|inr|₹)\s*([\d,]+(?:\.\d+)?)\s+per missed emi",
            r"penalty of (?:rs\.?|inr|₹)\s*([\d,]+(?:\.\d+)?)",
        ]))

        d.bounce_charge = rupees(first_amount(t, [
            *self._inr_patterns("bounce charge", "dishonour charge", "ecs bounce", "mandate bounce"),
        ]))

        d.penal_interest_percent_monthly = first_amount(t, [
            r"([\d.]+)\s*%\s*per\s*(?:calendar\s*)?month\s*penal",
            r"penal interest[^%]{0,60}?([\d.]+)\s*%\s*per\s*month",
            r"([\d.]+)\s*%\s*p\.m\.\s*penal",
        ])

        d.penal_interest_percent_annual = first_amount(t, [
            r"penal interest[^%]{0,60}?([\d.]+)\s*%\s*(?:per\s*)?(?:annum|p\.a\.)",
            r"([\d.]+)\s*%\s*p\.a\.\s*penal",
        ])

        if contains_any(t, [r"\bno prepayment penalty\b", r"\bno pre-?payment penalty\b", r"\bnil prepayment\b"]):
            d.has_prepayment_penalty = False
        elif contains_any(t, [r"\bprepayment penalty\b", r"\bpre-?payment penalty\b", r"\bprepayment charge\b"]):
            d.has_prepayment_penalty = True

        if contains_any(t, [r"\bno foreclosure charges?\b", r"\bforeclosure[: ]+nil\b"]):
            d.has_foreclosure_charges = False
        elif contains_any(t, [r"\bforeclosure charges?\b", r"\bforeclosure fee\b"]):
            d.has_foreclosure_charges = True

        if contains_any(t, [
            r"\bno hidden charges\b",
            r"\bother charges\s*nil\b",
            r"\bno clause is hidden\b",
            r"\bnil hidden charges\b",
            r"\bno additional charges\b",
        ]):
            d.has_hidden_charges = False
        elif contains_any(t, [
            r"additional administrative",
            r"administrative or service charges",
            r"service charges during the loan tenure",
            r"other charges may apply",
            r"charges as deemed necessary",
            r"at the lender'?s discretion",
        ]):
            d.has_hidden_charges = True

        if contains_any(t, [r"auto[- ]?debit\s*\(optional\)", r"no mandatory auto[- ]?debit"]):
            d.auto_debit_optional = True
            d.auto_debit_mandatory = False
        elif contains_any(t, [
            r"emi shall be auto[- ]?debited",
            r"shall be auto[- ]?debited",
            r"auto[- ]?debit mandate",
            r"maintain sufficient balance for emi auto[- ]?debit",
            r"standing instruction.*?emi",
        ]):
            d.auto_debit_mandatory = True
            d.auto_debit_optional = False

        if contains_any(t, [
            r"loan protection insurance",
            r"credit life insurance",
            r"premium.*?added to the loan amount",
            r"insurance clause",
        ]):
            d.has_insurance_addon = True

        if contains_any(t, [
            r"interest rate.*?may be revised",
            r"floating rate",
            r"variable rate",
            r"rate reset",
            r"interest rate revision",
        ]):
            d.has_rate_reset_clause = True

        if contains_any(t, [
            r"legal (?:cost|fee|charges?).*?borrower shall bear",
            r"borrower shall bear.*?legal",
            r"legal expenses.*?charged to borrower",
        ]):
            d.has_legal_cost_transfer = True

        if contains_any(t, [
            r"cross[- ]?default",
            r"event of default.*?other loan",
            r"default on any other",
        ]):
            d.has_cross_default_clause = True

        return d

    def merge(self, ai_raw: Dict[str, Any], text: str) -> ExtractedData:
        ai = ExtractedData(**{
            field: ai_raw.get(field, getattr(ExtractedData(), field))
            for field in ExtractedData.__dataclass_fields__.keys()
        })
        hints = self.extract_rule_hints(text)
        t = text.lower()

        numeric_fields = [
            "loan_amount", "emi_amount", "interest_rate_percent", "tenure_months",
            "processing_fee_percent", "processing_fee_amount",
            "prepayment_penalty_percent", "foreclosure_charge_percent",
            "late_payment_fee", "bounce_charge",
            "penal_interest_percent_monthly", "penal_interest_percent_annual",
        ]
        for key in numeric_fields:
            if not getattr(ai, key):
                setattr(ai, key, getattr(hints, key))

        if re.search(r"no prepayment penalty|no pre-?payment penalty|nil prepayment", t):
            ai.has_prepayment_penalty = False
            ai.prepayment_penalty_percent = 0.0

        if re.search(r"no foreclosure charges?|foreclosure[: ]+nil", t):
            ai.has_foreclosure_charges = False
            ai.foreclosure_charge_percent = 0.0

        if re.search(r"no hidden charges|other charges\s*nil|nil hidden charges|no additional charges", t):
            ai.has_hidden_charges = False
        elif not ai.has_hidden_charges and hints.has_hidden_charges:
            ai.has_hidden_charges = True

        if re.search(r"auto.?debit\s*\(optional\)|no mandatory auto.?debit", t):
            ai.auto_debit_mandatory = False
            ai.auto_debit_optional = True

        if not ai.has_prepayment_penalty and hints.has_prepayment_penalty and not re.search(r"no prepayment penalty|no pre-?payment penalty", t):
            ai.has_prepayment_penalty = True
            if not ai.prepayment_penalty_percent:
                ai.prepayment_penalty_percent = hints.prepayment_penalty_percent

        if not ai.has_foreclosure_charges and hints.has_foreclosure_charges and not re.search(r"no foreclosure charges?", t):
            ai.has_foreclosure_charges = True
            if not ai.foreclosure_charge_percent:
                ai.foreclosure_charge_percent = hints.foreclosure_charge_percent

        if not ai.auto_debit_mandatory and hints.auto_debit_mandatory and not ai.auto_debit_optional:
            ai.auto_debit_mandatory = True

        if not ai.has_insurance_addon and hints.has_insurance_addon:
            ai.has_insurance_addon = True
        if not ai.has_rate_reset_clause and hints.has_rate_reset_clause:
            ai.has_rate_reset_clause = True
        if not ai.has_legal_cost_transfer and hints.has_legal_cost_transfer:
            ai.has_legal_cost_transfer = True
        if not ai.has_cross_default_clause and hints.has_cross_default_clause:
            ai.has_cross_default_clause = True

        return ai


# MONETARY ENGINE
class MonetaryEngine:
    def estimate(self, data: ExtractedData) -> CostBreakdown:
        loan = data.loan_amount
        emi = data.emi_amount

        if data.processing_fee_amount:
            processing_fee_cost = data.processing_fee_amount
        elif data.processing_fee_percent and loan:
            processing_fee_cost = rupees(loan * data.processing_fee_percent / 100)
        else:
            processing_fee_cost = 0

        prepayment_cost = rupees(loan * data.prepayment_penalty_percent / 100) if loan and data.prepayment_penalty_percent else 0
        foreclosure_cost = rupees(loan * data.foreclosure_charge_percent / 100) if loan and data.foreclosure_charge_percent else 0

        late_payment_cost_estimate = data.late_payment_fee * cfg.LATE_MISS_ESTIMATE
        bounce_charge_cost_estimate = data.bounce_charge * cfg.BOUNCE_MISS_ESTIMATE

        if data.penal_interest_percent_monthly and emi:
            penal_interest_cost_estimate = rupees(
                emi * (data.penal_interest_percent_monthly / 100) * cfg.PENAL_OVERDUE_MONTHS
            )
        elif data.penal_interest_percent_annual and emi:
            monthly_rate = data.penal_interest_percent_annual / 12
            penal_interest_cost_estimate = rupees(
                emi * (monthly_rate / 100) * cfg.PENAL_OVERDUE_MONTHS
            )
        else:
            penal_interest_cost_estimate = 0

        insurance_addon_cost_estimate = rupees(loan * cfg.INSURANCE_RATE) if (data.has_insurance_addon and loan) else 0
        hidden_charges_estimate = cfg.HIDDEN_CHARGES_ESTIMATE if data.has_hidden_charges else 0

        total_estimated_cost = (
            processing_fee_cost
            + prepayment_cost
            + foreclosure_cost
            + late_payment_cost_estimate
            + bounce_charge_cost_estimate
            + penal_interest_cost_estimate
            + insurance_addon_cost_estimate
            + hidden_charges_estimate
        )

        return CostBreakdown(
            processing_fee_cost=processing_fee_cost,
            prepayment_cost=prepayment_cost,
            foreclosure_cost=foreclosure_cost,
            late_payment_cost_estimate=late_payment_cost_estimate,
            bounce_charge_cost_estimate=bounce_charge_cost_estimate,
            penal_interest_cost_estimate=penal_interest_cost_estimate,
            insurance_addon_cost_estimate=insurance_addon_cost_estimate,
            hidden_charges_estimate=hidden_charges_estimate,
            total_estimated_cost=total_estimated_cost,
        )


# CLAUSE BUILDER
class ClauseBuilder:
    def build(self, data: ExtractedData, costs: CostBreakdown, text: str) -> List[Clause]:
        clauses: List[Clause] = []

        # ── FIX 3: downgrade High → Medium when financial_impact is ₹0 ──
        def add(title: str, clause_type: str, severity: str, meaning: str, impact: int, recommendation: str, patterns: List[str], confidence: float = 0.96):
            if impact == 0 and severity == "High":
                severity = "Medium"
            clauses.append(Clause(
                title=title,
                type=clause_type,
                severity=severity,
                plain_language_en=meaning,
                financial_impact_rupees=impact,
                recommendation=recommendation,
                evidence_text=extract_snippet(text, patterns),
                confidence=confidence,
            ))

        if costs.processing_fee_cost:
            severity = "Low" if costs.processing_fee_cost <= 1000 else "Medium" if costs.processing_fee_cost <= 5000 else "High"
            add(
                "Processing Fee", "Upfront Cost", severity,
                "You may have to pay an upfront processing fee before or during loan disbursement.",
                costs.processing_fee_cost,
                "Ask whether this fee is negotiable and compare with lower-cost lenders.",
                [r"processing fee", r"processing fees", r"processing charge"],
            )

        if data.has_prepayment_penalty:
            severity = "High" if costs.prepayment_cost > 0 else "Medium"
            amount_desc = f"Closing the loan early could cost about ₹{costs.prepayment_cost:,}." if costs.prepayment_cost else "Closing the loan early may lead to an additional penalty."
            add(
                "Prepayment Penalty", "Penalty", severity,
                f"You may be charged extra if you repay the loan before the full tenure. {amount_desc}",
                costs.prepayment_cost,
                "Negotiate removal of this clause or avoid early closure without checking the cost.",
                [r"prepayment penalty", r"pre-?payment penalty", r"prepayment charge"], 0.98,
            )

        if data.has_foreclosure_charges:
            amount_desc = f"Foreclosing the loan may cost around ₹{costs.foreclosure_cost:,}." if costs.foreclosure_cost else "Foreclosing the loan may attract extra charges."
            add(
                "Foreclosure Charges", "Penalty", "High",
                f"You may be charged for fully closing the loan before tenure ends. {amount_desc}",
                costs.foreclosure_cost,
                "Ask for a foreclosure-free product or get the lender to waive this charge.",
                [r"foreclosure charge", r"foreclosure charges", r"foreclosure fee"], 0.98,
            )

        if data.auto_debit_mandatory:
            add(
                "Auto-Debit Mandate", "Control Restriction", "Medium",
                "The lender requires your EMI to be auto-debited from your bank account, reducing payment flexibility.",
                0,
                "Keep a buffer in the linked account and understand mandate failure penalties.",
                [r"auto-?debit mandate", r"shall be auto-?debited", r"auto-?debit"], 0.96,
            )

        if data.late_payment_fee:
            severity = "Low" if data.late_payment_fee <= 300 else "Medium" if data.late_payment_fee <= 750 else "High"
            add(
                "Late Payment Conditions", "Penalty", severity,
                f"If you miss or delay an EMI, a fee of about ₹{data.late_payment_fee:,} can be charged each time.",
                costs.late_payment_cost_estimate,
                "Use reminders and keep enough balance before the EMI date.",
                [r"late payment fee", r"late payment charge", r"per missed emi", r"overdue charge"], 0.97,
            )

        if data.bounce_charge:
            severity = "Medium" if data.bounce_charge <= 750 else "High"
            add(
                "Bounce Charges", "Penalty", severity,
                f"If auto-debit fails, the lender may charge about ₹{data.bounce_charge:,} per failed attempt.",
                costs.bounce_charge_cost_estimate,
                "Maintain enough balance in the linked account around the EMI date.",
                [r"bounce charge", r"dishonour charge", r"ecs bounce"], 0.97,
            )

        if data.penal_interest_percent_monthly or data.penal_interest_percent_annual:
            if data.penal_interest_percent_monthly:
                rate_text = f"{data.penal_interest_percent_monthly}% per month"
                severity = "High" if data.penal_interest_percent_monthly >= 2 else "Medium"
            else:
                rate_text = f"{data.penal_interest_percent_annual}% per annum"
                severity = "High" if data.penal_interest_percent_annual >= 24 else "Medium"
            add(
                "Penal Interest", "Penalty", severity,
                f"If EMIs are overdue, extra penal interest at about {rate_text} may be charged on overdue amounts.",
                costs.penal_interest_cost_estimate,
                "Avoid rolling overdue EMIs because extra interest can increase your burden quickly.",
                [r"penal interest", r"overdue interest", r"default interest"], 0.98,
            )

        if data.has_hidden_charges:
            add(
                "Hidden Processing / Administrative Charges", "Hidden Cost", "High",
                "The agreement allows additional service or administrative charges that are not fully specified upfront.",
                costs.hidden_charges_estimate,
                "Ask for a written schedule of all charges before signing.",
                [r"administrative charges", r"service charges", r"charges as deemed necessary", r"additional administrative"], 0.99,
            )

        if data.has_insurance_addon:
            add(
                "Insurance Add-on", "Add-on Cost", "Medium",
                "The loan may include an insurance premium that increases the total amount you pay.",
                costs.insurance_addon_cost_estimate,
                "Check whether this insurance is optional and compare with standalone insurance.",
                [r"loan protection insurance", r"credit life insurance", r"insurance clause"], 0.95,
            )

        if data.has_rate_reset_clause:
            add(
                "Floating / Revisable Interest Rate", "Rate Risk", "Medium",
                "The interest rate may change later, which can increase your EMI or your loan tenure.",
                0,
                "Ask whether there is a fixed-rate option or rate cap.",
                [r"floating rate", r"variable rate", r"rate reset", r"interest rate.*may be revised"], 0.93,
            )

        if data.has_legal_cost_transfer:
            add(
                "Legal Cost Transfer", "Liability Transfer", "High",
                "If recovery or legal action happens, legal expenses may be passed on to you.",
                10000,
                "Understand default conditions and try to cap borrower-side legal liability.",
                [r"legal cost", r"legal charges", r"legal expenses.*borrower"], 0.93,
            )

        if data.has_cross_default_clause:
            add(
                "Cross-Default Clause", "Risk Amplifier", "High",
                "Default on another loan may automatically trigger default under this loan too.",
                0,
                "Review this clause carefully before signing because it can increase your overall risk sharply.",
                [r"cross-?default", r"default on any other", r"event of default.*other loan"], 0.94,
            )

        clauses.sort(key=lambda c: severity_rank(c.severity))
        return clauses


# OVERALL RISK SCORING
class RiskScorer:
    WEIGHTS = {"High": 18, "Medium": 8, "Low": 3}

    def score(self, clauses: List[Clause], costs: CostBreakdown) -> Tuple[int, str, str]:
        raw = sum(self.WEIGHTS.get(c.severity, 0) for c in clauses)

        total_cost = costs.total_estimated_cost
        if total_cost > 100000:
            raw += 20
        elif total_cost > 50000:
            raw += 15
        elif total_cost > 20000:
            raw += 10
        elif total_cost > 5000:
            raw += 5
        elif total_cost > 2000:
            raw += 2

        score = int(clamp(raw, 0, 100))

        if score <= 10:
            level, summary = "Very Low", "This agreement appears borrower-friendly with minimal financial risk."
        elif score <= 30:
            level, summary = "Low", "This agreement has limited risk, but a few terms still deserve attention."
        elif score <= 60:
            level, summary = "Moderate", "This agreement has meaningful financial risks. Review the highlighted clauses carefully before signing."
        elif score <= 80:
            level, summary = "High", "This agreement contains several costly or restrictive clauses and should be reviewed carefully."
        else:
            level, summary = "Critical", "This agreement is highly restrictive or financially risky and should not be signed without careful review."

        return score, level, summary


# RISK REASONS — specific field-driven list
def build_risk_reasons(data: ExtractedData, costs: CostBreakdown) -> List[str]:
    """Returns a plain-English list of exactly what drove the risk score."""
    risk_reasons: List[str] = []

    if data.has_prepayment_penalty:
        risk_reasons.append(f"Prepayment penalty ₹{costs.prepayment_cost:,}")

    if data.has_hidden_charges:
        risk_reasons.append("Hidden charges present")

    if data.auto_debit_mandatory:
        risk_reasons.append("Auto-debit mandatory")

    if costs.total_estimated_cost > 20000:
        risk_reasons.append("High total financial cost")

    if not risk_reasons:
        risk_reasons.append("No major risk factors detected.")

    return risk_reasons


# AI SUMMARY REASONING
class AISummaryEngine:
    def summarize(self, data: ExtractedData, costs: CostBreakdown, clauses: List[Clause], score: int, level: str) -> str:
        compact = {
            "loan_amount": data.loan_amount,
            "emi_amount": data.emi_amount,
            "processing_fee_percent": data.processing_fee_percent,
            "processing_fee_amount": data.processing_fee_amount,
            "prepayment_penalty_percent": data.prepayment_penalty_percent,
            "foreclosure_charge_percent": data.foreclosure_charge_percent,
            "late_payment_fee": data.late_payment_fee,
            "bounce_charge": data.bounce_charge,
            "penal_interest_percent_monthly": data.penal_interest_percent_monthly,
            "has_hidden_charges": data.has_hidden_charges,
            "auto_debit_mandatory": data.auto_debit_mandatory,
            "total_estimated_cost": costs.total_estimated_cost,
            "clause_titles": [c.title for c in clauses],
            "score": score,
            "level": level,
        }

        messages = [
            {
                "role": "system",
                "content": (
                    "You are a financial advisor AI for Indian borrowers. "
                    "Write EXACTLY 2 sentences summarizing this loan's risk. "
                    "Sentence 1: clearly state the risk level and score. "
                    "Sentence 2: mention ONLY the most important risky clause and its estimated total cost in ₹ (Indian Rupees). "
                    "STRICT RULES: "
                    "Always use ₹ symbol, never $. "
                    "All costs are TOTAL costs, not monthly, unless explicitly stated. "
                    "Do NOT invent concepts like 'allowed monthly cost' or similar. "
                    "If there are no high-risk clauses, mention the most relevant available clause or say the loan is generally safe. "
                    "Do NOT mention processing fee if it is 0 or not applicable. "
                    "Do NOT contradict yourself — risk level must match the score. "
                    "No extra sentences. No bullet points. No padding."
                ),
            },
            {
                "role": "user",
                "content": json.dumps(compact, ensure_ascii=False),
            },
        ]

        try:
            raw = normalize(ModelSingleton.generate(messages, max_new_tokens=120))
            raw = raw.replace("$", "₹")
            return raw
        except Exception:
            if clauses:
                top = clauses[0]
                impact = f"₹{top.financial_impact_rupees:,}" if top.financial_impact_rupees else "an unknown amount"
                return (
                    f"This agreement scores {score}/100 and is rated {level}. "
                    f"The biggest concern is {top.title.lower()}, with an estimated cost of {impact}."
                )
            return (
                f"This agreement scores {score}/100 and is rated {level}. "
                f"No major risky clauses were detected."
            )


# REPORT ENGINE
class ReportEngine:
    def build(self, result: AnalysisResult) -> str:
        lines: List[str] = []
        lines.append("LOANGUARD AI — DECISION-READY SUMMARY REPORT")
        lines.append("=" * 52)
        lines.append(f"Document      : {result.document_name}")
        lines.append(f"Risk Score    : {result.overall_score}/100")
        lines.append(f"Risk Level    : {result.overall_risk_level}")
        lines.append(f"Confidence    : {result.confidence_score * 100:.0f}%")
        lines.append(f"Hidden Cost   : ₹{result.total_hidden_cost_estimate_rupees:,}")
        lines.append("")
        lines.append("Overall Assessment:")
        lines.append(result.overall_summary)
        lines.append("")
        lines.append("Why This Score:")
        for reason in result.risk_reasons:
            lines.append(f"  • {reason}")
        lines.append("")
        lines.append("Detected Clauses:")

        if not result.clauses:
            lines.append("- No risky clauses detected")
        else:
            for idx, c in enumerate(result.clauses, start=1):
                lines.append(f"{idx}. {c['title']}")
                lines.append(f"   Type     : {c['type']}")
                lines.append(f"   Severity : {c['severity']}")
                lines.append(f"   Meaning  : {c['plain_language_en']}")
                lines.append(f"   Impact   : ₹{c['financial_impact_rupees']:,}")
                lines.append(f"   Action   : {c['recommendation']}")
                lines.append("")

        return "\n".join(lines)


# CONFIDENCE
def compute_confidence(data: ExtractedData, clauses: List[Clause]) -> float:
    score = 0.55
    if data.loan_amount:       score += 0.08
    if data.emi_amount:        score += 0.05
    if data.interest_rate_percent: score += 0.04
    if data.tenure_months:     score += 0.03
    if data.late_payment_fee:  score += 0.04
    if data.bounce_charge:     score += 0.04
    if data.processing_fee_percent or data.processing_fee_amount: score += 0.04
    if data.auto_debit_mandatory or data.auto_debit_optional:     score += 0.04
    if clauses:                score += 0.05
    return round(min(score, 0.99), 2)


# MAIN ANALYSIS
def analyze_document(pdf_bytes: bytes, filename: str) -> AnalysisResult:
    t0 = time.perf_counter()
    warnings: List[str] = []

    text = extract_pdf_text(pdf_bytes)
    ai_extractor = AIExtractor()
    validator = ValidationEngine()

    try:
        ai_raw = ai_extractor.extract(text)
    except Exception as exc:
        log.warning("AI extraction failed for %s: %s", filename, exc)
        warnings.append(f"AI extraction partially failed. Falling back to rule layer. ({exc})")
        ai_raw = {}

    data = validator.merge(ai_raw, text)
    costs = MonetaryEngine().estimate(data)
    clauses = ClauseBuilder().build(data, costs, text)
    score, level, fallback_summary = RiskScorer().score(clauses, costs)

    summary = AISummaryEngine().summarize(data, costs, clauses, score, level) or fallback_summary

    risk_reasons = build_risk_reasons(data, costs)

    confidence = compute_confidence(data, clauses)

    metrics = {
        "detected_clauses": len(clauses),
        "high_risk_clauses": sum(1 for c in clauses if c.severity == "High"),
        "medium_risk_clauses": sum(1 for c in clauses if c.severity == "Medium"),
        "low_risk_clauses": sum(1 for c in clauses if c.severity == "Low"),
    }

    borrower_signal = (
        "Good choice for borrower" if score <= 20
        else "Review before signing" if score <= 60
        else "Seek legal advice before signing"
    )

    result = AnalysisResult(
        document_name=filename,
        processing_time_sec=round(time.perf_counter() - t0, 2),
        confidence_score=confidence,
        overall_score=score,
        overall_risk_level=level,
        overall_summary=summary,
        total_hidden_cost_estimate_rupees=costs.total_estimated_cost,
        borrower_signal=borrower_signal,
        risk_reasons=risk_reasons,
        metrics=metrics,
        extracted_data=asdict(data),
        cost_breakdown=asdict(costs),
        clauses=[asdict(c) for c in clauses],
        warnings=warnings,
    )

    result.decision_ready_report = ReportEngine().build(result)
    return result


# FLASK APP
app = Flask(__name__)
CORS(app)
app.config["MAX_CONTENT_LENGTH"] = cfg.MAX_UPLOAD_MB * 1024 * 1024


def error_response(message: str, status: int = 400):
    return jsonify({"error": message}), status


@app.errorhandler(413)
def too_large(_):
    return error_response(f"File too large. Maximum upload size is {cfg.MAX_UPLOAD_MB} MB.", 413)


@app.route("/")
def health():
    return jsonify({
        "status": "ok",
        "service": "LoanGuard AI Backend",
        "version": "4.2",
        "model": cfg.MODEL_NAME,
        "device": ModelSingleton._device,
    })


@app.route("/analyze", methods=["POST"])
def analyze():
    if "file" not in request.files:
        return error_response("No file uploaded. Use form-data field 'file'.")

    uploaded = request.files["file"]
    if not uploaded.filename or not uploaded.filename.lower().endswith(".pdf"):
        return error_response("Only PDF files are supported.")

    try:
        result = analyze_document(uploaded.read(), uploaded.filename)
        return jsonify(result.to_dict())
    except ValueError as exc:
        return error_response(str(exc), 422)
    except Exception:
        log.error("Unhandled error in /analyze:\n%s", traceback.format_exc())
        return error_response("Internal analysis error. Check backend logs.", 500)


@app.route("/compare", methods=["POST"])
def compare():
    files = request.files.getlist("files")
    if len(files) < 2:
        return error_response("Upload at least 2 PDF files for comparison.")
    if len(files) > 5:
        return error_response("Maximum 5 PDF files supported for comparison.")

    results: List[AnalysisResult] = []
    skipped_errors: List[str] = []

    for f in files:
        try:
            if not f.filename.lower().endswith(".pdf"):
                skipped_errors.append(f"{f.filename}: not a PDF")
                continue
            results.append(analyze_document(f.read(), f.filename))
        except Exception as exc:
            skipped_errors.append(f"{f.filename}: {exc}")

    if len(results) < 2:
        return error_response("Could not analyze enough files for comparison.", 422)

    ranked = sorted(
        results,
        key=lambda r: (r.overall_score, r.total_hidden_cost_estimate_rupees)
    )

    best = ranked[0]

    runner_up = ranked[1]
    score_diff = runner_up.overall_score - best.overall_score
    cost_diff = runner_up.total_hidden_cost_estimate_rupees - best.total_hidden_cost_estimate_rupees
    recommendation_reason = (
        f"Lowest risk score ({best.overall_score}/100) "
        f"and lowest estimated hidden cost (₹{best.total_hidden_cost_estimate_rupees:,}). "
        f"Saves approximately ₹{cost_diff:,} and scores {score_diff} points better than the next option."
        if cost_diff >= 0 else
        f"Lowest risk score ({best.overall_score}/100) among all uploaded agreements."
    )

    comparison = [
        {
            "rank": idx + 1,
            "document_name": r.document_name,
            "overall_score": r.overall_score,
            "overall_risk_level": r.overall_risk_level,
            "estimated_hidden_cost": r.total_hidden_cost_estimate_rupees,
            "clause_count": r.metrics["detected_clauses"],
            "high_risk_clauses": r.metrics["high_risk_clauses"],
            "borrower_signal": r.borrower_signal,
            "confidence_score": r.confidence_score,
            "confidence_percent": f"{r.confidence_score * 100:.0f}%",
            "risk_reasons": r.risk_reasons,
        }
        for idx, r in enumerate(ranked)
    ]

    return jsonify({
        "recommended_loan": {
            "document_name": best.document_name,
            "overall_score": best.overall_score,
            "overall_risk_level": best.overall_risk_level,
            "estimated_hidden_cost": best.total_hidden_cost_estimate_rupees,
            "confidence_score": best.confidence_score,
            "confidence_percent": f"{best.confidence_score * 100:.0f}%",
            "reason": recommendation_reason,
        },
        "comparison_table": comparison,
        "documents": [r.to_dict() for r in ranked],
        "skipped_errors": skipped_errors,
    })


#START BACKEND
def start_backend():
    ModelSingleton.load()

    public_url = None

    if cfg.NGROK_TOKEN:
        try:
            ngrok.set_auth_token(cfg.NGROK_TOKEN)
            tunnel = ngrok.connect(cfg.PORT)
            public_url = tunnel.public_url

            log.info("🔥 Public URL: %s", public_url)
            print("\n✅ Backend is live at:", public_url)
            print("📌 Paste this into your frontend API_URL\n")

        except Exception as e:
            log.error("Ngrok failed: %s", str(e))
            print("\n❌ Ngrok failed. Running locally only.\n")

    else:
        log.warning("NGROK_AUTHTOKEN not set — running locally only.")
        print("\n⚠️ No NGROK token found. Backend running locally.\n")

    print(f"🚀 Starting server on port {cfg.PORT}...\n")
    app.run(host="0.0.0.0", port=cfg.PORT, debug=False)


# Run backend
start_backend()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


✅ Backend is live at: https://unteaseled-potentially-urijah.ngrok-free.dev
📌 Paste this into your frontend API_URL

🚀 Starting server on port 5000...

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [29/Mar/2026 05:52:39] "POST /analyze HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [29/Mar/2026 05:52:49] "POST /analyze HTTP/1.1" 200 -
